# 국어나래 내장 코치 모델 파인튜닝 (Gemma 3 1B)

학생 기기 안에서 돌아가는 '생각 코치' 모델을 학습시킵니다. **학생에게 해가 되는 답을 하지 않는 것**이 최우선 목표입니다.

## 실행 전 준비

**공통**
- Hugging Face 토큰(Write 권한)을 `HF_TOKEN` 이름의 보안 비밀로 넣어 두세요(아래 플랫폼별 설명).
- 바로 아래 '설정' 칸의 `REPO`에 내 Hugging Face 아이디를 넣으세요.

**Google Colab**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`(무료)
2. 왼쪽 열쇠 아이콘(보안 비밀)에 `HF_TOKEN`을 넣고 '노트북 액세스' 켜기
3. 맨 위 칸을 실행하면 **Google Drive 연결 허용 창**이 떠요. '허용'을 눌러야 학습 결과가 `내 드라이브/narae-train/`에 저장돼요(거절하면 이 런타임에만 저장되어 끊기면 사라져요).

**Kaggle Notebooks**(무료 GPU 주 30시간 안팎, Colab 한도가 끝났을 때 대신 쓰세요)
1. 이 노트북을 GitHub 주소로 가져오기(`File → Import Notebook → GitHub`) 또는 파일 올리기
2. 오른쪽 `Settings`에서 Accelerator를 `GPU T4 x2`(또는 P100), `Internet`을 켜기(전화번호 인증이 필요해요)
3. 메뉴 `Add-ons → Secrets`에 `HF_TOKEN` 추가하고 이 노트북에 연결
4. 결과는 `/kaggle/working/narae-train/`에 쌓이고, **`Save Version`(Save & Run All)** 으로 저장해야 남아요. 오래 걸리니 `Save & Run All (Commit)`으로 돌려 두는 것이 안전해요.
5. 세션이 끊긴 뒤 이어 하려면 오른쪽 `Input → Add Input`으로 **내 이전 버전의 노트북 출력**을 붙이세요. 첫 칸이 거기서 `narae-train`을 자동으로 복사해 와요.

## 끊겼을 때 이어 하기

`런타임 → 모두 실행`(Kaggle은 `Run All`)을 다시 누르면 **끝난 단계는 건너뛰고**, 학습 중이던 SFT·DPO는 50단계마다 저장한 체크포인트부터 이어 해요. 체크포인트를 저장하는 동안(몇 초)만 아니면 언제 끊겨도 괜찮아요.

## 하는 일과 시간(대략, T4 기준)

| 단계 | 하는 일 | 시간 |
| --- | --- | --- |
| 0 | 학습 전 모델 시험(선택, 기본 꺼짐) | 20~30분 |
| 1 | SFT: 좋은 답(질문만)을 따라 하도록 학습 | 60~80분 |
| 2 | SFT 모델 안전 시험(950문항) | 20~30분 |
| 3 | DPO: '안전한 답'과 '해로운 답'을 비교해 안전한 쪽을 고르도록 학습 | 60~90분 |
| 4 | DPO 모델 안전 시험 | 20~30분 |
| 5 | 안전 기준을 통과한 모델 고르기(없으면 자동으로 멈춤) | 바로 |
| 6 | 앱 형식(ONNX int8)으로 변환 | 10~20분 |
| 7 | Hugging Face에 올리기 | 몇 분 |

학습 전 모델 시험은 비교 기준일 뿐이에요. 이전 결과에서 학습 전 모델은 안전 기준을 통과하지 못했으므로 모델을 채택하는 데 필요하지 않아 기본으로 껐어요(GPU 시간 절약).
SFT·DPO 시험이 끝나면 결과를 Hugging Face에 바로 올려요(`REPO`와 토큰이 있을 때). 세션이 중간에 끊겨도 검수자가 결과를 볼 수 있어요.

**안전 설계(학습 데이터)**: 대필 요청(우회·압박·역할극), 학생 글 속 숨은 지시문, 모욕·폭력·욕설·차별·사생활 침해가 담긴 글, 위험 신호(괴롭힘·학대·자해), 위험 신호처럼 보이지만 아닌 표현(거짓 알림 방지), 개인 정보, 주제 이탈, 맞춤법 정답 요구.
앱의 가드레일·규칙 감지기는 그대로 유지됩니다(학습은 추가 안전망).

데이터 출처: Claude가 작성한 질문 은행·가상 학생 글을 프로그램으로 조합(실제 학생 데이터 없음). 교사가 약관 확인 후 사용 결정(2026-11-03).

## 설정

In [ ]:
# ── 설정(여기만 고치면 돼요) ──
REPO = 'YOUR_HF_USERNAME/narae-coach-gemma3-1b'   # ← YOUR_HF_USERNAME 자리에 내 Username(프로필 주소의 아이디)
PRIVATE = False            # True면 비공개 저장소(받을 때 토큰 필요). 이미 있는 저장소의 공개 여부는 바꾸지 않아요.
RUN_BASE_EVAL = False      # 학습 전 모델 시험(비교용, 20~30분). 채택에는 필요 없어서 기본 꺼짐
EVAL_BATCH = 16            # 시험 때 한 번에 처리할 문항 수. T4에서 메모리 오류가 나면 8로 줄이세요
QUICK_EVAL_LIMIT = 0       # 0이면 950문항 전체. 숫자(예: 100)를 넣으면 앞 N개만 빨리 봐요(채택 판단에는 쓸 수 없어요)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. 준비: 저장 위치 정하기, 코드 받기
- Colab: Google Drive `narae-train/` / Kaggle: `/kaggle/working/narae-train/` / 그 밖: 현재 폴더의 `narae-train/`
- 학습 결과(`runs/`)와 변환 결과(`export/`)는 이 저장 위치에 쌓여요. 다시 실행해도 안전해요.

In [ ]:
import os, sys, shutil, glob, json, subprocess

# 명령이 실패하면 '모두 실행'이 거기서 멈추도록(그냥 !python 은 실패해도 다음 칸으로 넘어가요)
def sh(cmd):
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(f'명령이 실패했어요(코드 {proc.returncode}): {cmd}\n위 오류 메시지 마지막 부분을 Claude에게 알려 주세요.')

# 1) 어디서 돌고 있는지 알아내고, 저장 위치(WORK)를 정합니다.
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if os.path.exists('/kaggle'):
    PLATFORM = 'kaggle'
    WORK = '/kaggle/working/narae-train'
    CODE_ROOT = '/kaggle/working'
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = (UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN).strip()
    except Exception as e:
        print('Kaggle 보안 비밀(HF_TOKEN)을 못 읽었어요:', type(e).__name__, e)
    # 새 세션이면 이전 버전의 노트북 출력(Add Input)에서 학습 결과를 복사해 옵니다.
    if not os.path.isdir(os.path.join(WORK, 'runs')):
        found = sorted(glob.glob('/kaggle/input/*/narae-train') + glob.glob('/kaggle/input/*/*/narae-train'))
        if found:
            print('이전 결과를 가져와요:', found[0])
            shutil.copytree(found[0], WORK, dirs_exist_ok=True)
else:
    try:
        from google.colab import drive, userdata
        PLATFORM = 'colab'
    except Exception:
        PLATFORM = 'local'
    CODE_ROOT = '/content' if PLATFORM == 'colab' else os.getcwd()
    WORK = os.path.abspath('narae-train')
    if PLATFORM == 'colab':
        try:
            drive.mount('/content/drive')
            if not os.path.isdir('/content/drive/MyDrive'):
                raise RuntimeError('Drive 폴더가 보이지 않아요')
            WORK = '/content/drive/MyDrive/narae-train'
        except Exception as e:
            WORK = '/content/narae-train'
            print('!' * 60)
            print('경고: Google Drive를 연결하지 못했어요(' + type(e).__name__ + ': ' + str(e) + ').')
            print('학습 결과가 이 런타임에만 저장돼요. 끊기면 모두 사라지고 처음부터 다시 해야 해요.')
            print('Drive에 저장하려면 이 칸을 다시 실행하고 연결 허용 창에서 "허용"을 눌러 주세요.')
            print('!' * 60)
        try:
            HF_TOKEN = (userdata.get('HF_TOKEN') or HF_TOKEN).strip()
        except Exception as e:
            print('Colab 보안 비밀(HF_TOKEN)을 못 읽었어요:', type(e).__name__, e)
os.makedirs(WORK, exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
print('실행 환경:', PLATFORM, '/ 저장 위치:', WORK)

# 2) 학습 코드와 데이터 받기(공개 저장소). 이미 받았으면 최신본만 받아요.
os.chdir(CODE_ROOT)
if os.path.isdir('narae-kor-aidt'):
    sh('cd narae-kor-aidt && rm -rf training/__pycache__ && git pull -q')
else:
    sh('git clone -q --depth 1 -b feat/module-core https://github.com/oswprogram10-tech/narae-kor-aidt.git')
TRAIN = os.path.join(CODE_ROOT, 'narae-kor-aidt', 'training')
os.chdir(TRAIN)

# 3) runs/, export/를 저장 위치로 연결(심볼릭 링크). 코드는 runs/..., export/... 로만 적으면 돼요.
for name in ['runs', 'export']:
    target = os.path.join(WORK, name)
    link = os.path.join(TRAIN, name)
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        if os.path.realpath(link) != os.path.realpath(target):
            os.unlink(link)
    elif os.path.isdir(link):   # 링크 만들기 전에 생긴 진짜 폴더면 내용을 옮겨 둡니다
        shutil.copytree(link, target, dirs_exist_ok=True)
        shutil.rmtree(link)
    if not os.path.lexists(link):
        os.symlink(target, link)

# 4) 패키지 설치
sh('pip install -q -U transformers trl peft datasets accelerate huggingface_hub')
# Colab에 미리 깔린 옛 torchao(0.10)는 새 peft와 충돌해 학습이 멈춰요. 이 학습엔 필요 없어서 지웁니다.
sh('pip uninstall -y -q torchao || true')

# 5) 단계 건너뛰기·시험·결과 올리기 도우미
def exists(p):
    return os.path.exists(p)

REPO_OK = not REPO.startswith('YOUR' + '_HF_') and REPO.count('/') == 1

def run_eval(name, model):
    out = f'runs/{name}/redteam_outputs.jsonl'
    score = out + '.score.json'
    limited = QUICK_EVAL_LIMIT > 0
    if exists(score) and json.load(open(score, encoding='utf-8')).get('limited', False) == limited:
        print(f'[건너뜀] {name} 안전 시험은 이미 끝났어요:', score)
        return
    cmd = f'python finetune.py eval --model {model} --out {out} --batch {EVAL_BATCH}'
    if limited:
        cmd += f' --limit {QUICK_EVAL_LIMIT}'
    sh(cmd)

def upload_results(name):
    """시험 결과를 Hugging Face에 바로 올려 둡니다(세션이 끊겨도 검수할 수 있게). 실패해도 멈추지 않아요."""
    try:
        if not HF_TOKEN:
            print('[결과 올리기 건너뜀] HF_TOKEN이 없어요.')
            return
        if not REPO_OK:
            print('[결과 올리기 건너뜀] 설정 칸의 REPO에 내 Hugging Face 아이디를 넣어 주세요.')
            return
        sh(f'python upload_hf.py --repo {REPO} --results-only --names {name}' + (' --private' if PRIVATE else ''))
    except Exception as e:
        print('경고: 결과를 올리지 못했어요(학습은 계속해요):', type(e).__name__, e)

## 1. 학습 데이터 검수
선생님이 직접 몇 개를 읽어 보세요. 이상한 예가 있으면 학습 전에 알려 주세요(데이터 생성기: `training/gen/`).

In [ ]:
import json, random
print(open('data/stats.json', encoding='utf-8').read())
rows = [json.loads(l) for l in open('data/sft_train.jsonl', encoding='utf-8')]
random.seed(0)
for cat in ['help', 'ghostwrite', 'harmful_draft', 'distress', 'hard_negative', 'injection', 'personal', 'off_topic', 'spelling']:
    r = random.choice([x for x in rows if x['meta']['category'] == cat])
    print(f"\n[{cat}] {r['meta']['scenario']} / {r['meta']['stage']}")
    print('  학생:', r['messages'][1]['content'].split('학생 말:')[-1].split('\n')[0].strip())
    print('  목표 답:', r['messages'][2]['content'])

## 2. 학습 전 안전 시험(비교 기준, 선택)
학습하지 않은 Gemma 3 1B가 레드팀 문항(950개)에 어떻게 답하는지 봅니다. 비교용이라 기본으로 껐어요(`RUN_BASE_EVAL`). 이전 결과에서 학습 전 모델은 안전 기준을 통과하지 못했어요.

In [ ]:
if not RUN_BASE_EVAL:
    print('[건너뜀] 학습 전 모델 시험은 꺼져 있어요(RUN_BASE_EVAL = False). 채택에는 필요 없어요.')
else:
    run_eval('base', 'unsloth/gemma-3-1b-it')

## 3. SFT 학습(약 60~80분, 시험 20~30분)
50단계마다 저장해서, 끊겨도 다시 실행하면 이어서 해요.

In [ ]:
if exists('runs/sft/merged/config.json') and exists('runs/sft/DONE'):
    print('[건너뜀] SFT는 이미 끝났어요: runs/sft/merged')
else:
    sh('python finetune.py sft --base unsloth/gemma-3-1b-it --out runs/sft --epochs 2')

In [ ]:
run_eval('sft', 'runs/sft/merged')
upload_results('sft')

## 4. DPO 학습(안전한 답 선호, 약 60~90분, 시험 20~30분)

In [ ]:
if exists('runs/dpo/DONE'):
    print('[건너뜀] DPO는 이미 끝났어요: runs/dpo/merged')
else:
    sh('python finetune.py dpo --base runs/sft/merged --out runs/dpo')

In [ ]:
run_eval('dpo', 'runs/dpo/merged')
upload_results('dpo')

## 5. 채택할 모델 고르기(자동)
안전 시험에서 **모든 기준을 통과한 모델만** 고릅니다(DPO 우선, 그다음 SFT). 둘 다 불합격이면 여기서 멈춥니다. 그때는 위 시험 결과 표를 Claude에게 알려 주세요.

In [ ]:
import json, os
CHOSEN = None
for name in ['dpo', 'sft']:
    p = f'runs/{name}/redteam_outputs.jsonl.score.json'
    if os.path.exists(p):
        r = json.load(open(p, encoding='utf-8'))
        if r.get('limited'):
            print(name, '→ 일부 문항만 시험한 결과라 채택에 쓸 수 없어요(QUICK_EVAL_LIMIT = 0 으로 전체 시험을 다시 하세요)')
            continue
        print(name, '→', '통과' if r['passed'] else '불합격')
        if r['passed'] and CHOSEN is None:
            CHOSEN = f'runs/{name}/merged'
if CHOSEN is None:
    raise RuntimeError('안전 시험을 통과한 모델이 없어 여기서 멈춥니다. 변환·업로드는 하지 않습니다. 위 결과를 Claude에게 알려 주세요.')
print('채택:', CHOSEN)

## 6. 앱 형식(ONNX int8)으로 변환(약 10~20분)

In [ ]:
if exists('export/narae-gemma-1b/onnx/model_int8.onnx'):
    print('[건너뜀] 변환은 이미 끝났어요: export/narae-gemma-1b/onnx/model_int8.onnx')
else:
    sh('pip install -q optimum-onnx onnx onnxruntime')
    sh('python export_onnx.py --model ' + CHOSEN + ' --out export/narae-gemma-1b')
sh('ls -la export/narae-gemma-1b export/narae-gemma-1b/onnx')

## 7. Hugging Face에 올리기
1. https://huggingface.co 에 가입하고 Settings → Access Tokens에서 **Write** 토큰을 만드세요.
2. Colab은 왼쪽 열쇠 아이콘, Kaggle은 `Add-ons → Secrets`에 이름 `HF_TOKEN`으로 토큰을 넣으세요(Colab은 '노트북 액세스'도 켜기).
3. 맨 위 설정 칸의 `REPO`에 내 아이디를 넣었는지 확인하고 실행하세요. 학생 데이터는 들어 있지 않습니다. 공개(`PRIVATE=False`)로 두면 앱 설치 때 바로 받을 수 있고, 비공개로 하면 받을 때 토큰이 필요합니다. 이미 있는 저장소의 공개 여부는 바꾸지 않아요.

In [ ]:
import getpass
assert REPO_OK, '설정 칸 REPO의 YOUR_HF_USERNAME 자리에 내 Username을 넣어 주세요(예: hong/narae-coach-gemma3-1b). "Username/저장소이름" 모양이어야 해요'
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face 토큰(Write 권한)을 붙여 넣으세요: ').strip()
    os.environ['HF_TOKEN'] = HF_TOKEN
# 앞 단계 설치로 바뀐 패키지와 섞이지 않게 새 프로세스에서 올립니다.
sh('rm -rf __pycache__ && git pull -q')   # upload_hf.py 최신본 받기(캐시 파일이 받기를 막지 않게 먼저 지움)
sh(f'python upload_hf.py --repo {REPO}' + (' --private' if PRIVATE else ''))

## 끝
저장소 주소(예: `내아이디/narae-coach-gemma3-1b`)를 Claude에게 알려 주세요. 앱과 같은 코드로 안전 시험을 다시 채점하고, 브라우저에서 속도·품질을 확인한 뒤 앱의 기본 내장 모델로 연결합니다.